In [1]:
# # Dataset Generator for Composite DNA - Extended Version
# ## Supports Multiple Error Models: Erlich (EZ17), Grass (G15), Organick (O17)
# ## With Variable Sequence Lengths

# In[1]:

# =============================================================================
# CELL 1: IMPORTS
# =============================================================================
import random
import numpy as np
import pickle
import json
import os
from collections import Counter
from datetime import datetime
import time


In [2]:
# In[2]:

# =============================================================================
# CELL 2: CONFIGURATION
# =============================================================================

# ------------------- SELECT ERROR MODEL -------------------
# Options: "erlich", "grass", "organick"
ERROR_MODEL = "erlich"  # <-- CHANGE THIS TO SELECT ERROR MODEL

# ------------------- SELECT ALPHABET MODE -------------------
# Options: "2mix_only", "2mix_3mix", "2mix_3mix_4mix"
ALPHABET_MODE = "2mix_only"  # <-- CHANGE THIS TO SELECT ALPHABET MODE
# ------------------------------------------------------------

# Error model specifications
ERROR_MODEL_SPECS = {
    "erlich": {
        "full_length": 152,
        "index_length": 16,
        "seq_length": 136,  # 152 - 16
        "name": "EZ17"
    },
    "grass": {
        "full_length": 117,
        "index_length": 13,
        "seq_length": 104,  # 117 - 13
        "name": "G15"
    },
    "organick": {
        "full_length": 110,
        "index_length": 33,
        "seq_length": 77,   # 110 - 33
        "name": "O17"
    }
}

CONFIG = {
    # Error Model
    "error_model": ERROR_MODEL,
    "error_specs": ERROR_MODEL_SPECS[ERROR_MODEL],
    
    # Alphabet Mode
    "alphabet_mode": ALPHABET_MODE,
    
    # Dataset Parameters
    "num_samples": 100000,
    "seq_length": ERROR_MODEL_SPECS[ERROR_MODEL]["seq_length"],
    "max_coverage": 25,
    
    # Output Directory
    "dataset_dir": "./dataset",
    
    # Reproducibility
    "seed": 42
}

# Set vocab_size based on alphabet mode
VOCAB_SIZES = {
    "2mix_only": 10,
    "2mix_3mix": 14,
    "2mix_3mix_4mix": 15
}

CONFIG["vocab_size"] = VOCAB_SIZES[CONFIG["alphabet_mode"]]

# Create dataset name including error model
dataset_name = f"dna_{CONFIG['error_specs']['name']}_{CONFIG['alphabet_mode']}"
CONFIG["dataset_path"] = (f"{CONFIG['dataset_dir']}/"
                          f"{dataset_name}_"
                          f"{CONFIG['num_samples']}_{CONFIG['max_coverage']}.pkl")

os.makedirs(CONFIG['dataset_dir'], exist_ok=True)

print(f"{'='*60}")
print(f"🔋 DATASET GENERATION CONFIGURATION")
print(f"{'='*60}")
print(f"   Error Model: {CONFIG['error_model'].upper()} ({CONFIG['error_specs']['name']})")
print(f"   Sequence Length: {CONFIG['seq_length']}")
print(f"   Alphabet Mode: {CONFIG['alphabet_mode']}")
print(f"   Vocab Size: {CONFIG['vocab_size']} classes")
print(f"   Num Samples: {CONFIG['num_samples']:,}")
print(f"   Max Coverage: {CONFIG['max_coverage']}")
print(f"   Output Path: {CONFIG['dataset_path']}")
print(f"{'='*60}")

🔋 DATASET GENERATION CONFIGURATION
   Error Model: ERLICH (EZ17)
   Sequence Length: 136
   Alphabet Mode: 2mix_only
   Vocab Size: 10 classes
   Num Samples: 100,000
   Max Coverage: 25
   Output Path: ./dataset/dna_EZ17_2mix_only_100000_25.pkl


In [3]:
# =============================================================================
# CELL 3: SEED FOR REPRODUCIBILITY
# =============================================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

set_seed(CONFIG['seed'])
print(f"🎲 Random seed set to: {CONFIG['seed']}")

🎲 Random seed set to: 42


In [4]:
# =============================================================================
# CELL 4: COMPOSITE DNA ALPHABET DEFINITIONS
# =============================================================================

# ----- Pure Bases -----
PURE_BASES = {
    'A': ['A'],
    'C': ['C'],
    'G': ['G'],
    'T': ['T'],
}

# ----- Two-Nucleotide Mixtures (6 total) -----
# All C(4,2) = 6 combinations with uniform 0.5/0.5 distribution
TWO_MIX_MAP = {
    'M1': ['A', 'T'],  # A|T
    'M2': ['C', 'G'],  # C|G
    'M3': ['C', 'T'],  # C|T
    'M4': ['G', 'T'],  # G|T
    'M5': ['A', 'C'],  # A|C
    'M6': ['A', 'G'],  # A|G
}

# ----- Three-Nucleotide Mixtures (4 total) -----
# All C(4,3) = 4 combinations with uniform 1/3 each distribution
THREE_MIX_MAP = {
    'T1': ['A', 'C', 'G'],  # A|C|G
    'T2': ['A', 'C', 'T'],  # A|C|T
    'T3': ['A', 'G', 'T'],  # A|G|T
    'T4': ['C', 'G', 'T'],  # C|G|T
}

# ----- Four-Nucleotide Mixture (1 total) -----
# C(4,4) = 1 combination with uniform 0.25 each distribution
FOUR_MIX_MAP = {
    'Q1': ['A', 'C', 'G', 'T'],  # A|C|G|T
}


def build_composite_map(mode):
    """Build the composite symbol mapping based on alphabet mode."""
    composite_map = PURE_BASES.copy()
    composite_map.update(TWO_MIX_MAP)
    
    if mode in ["2mix_3mix", "2mix_3mix_4mix"]:
        composite_map.update(THREE_MIX_MAP)
    
    if mode == "2mix_3mix_4mix":
        composite_map.update(FOUR_MIX_MAP)
    
    return composite_map


def build_symbol_to_idx(mode):
    """Build symbol-to-index mapping based on alphabet mode."""
    # Pure bases: indices 0-3
    symbol_to_idx = {
        'A': 0, 'C': 1, 'G': 2, 'T': 3,
    }
    
    # Two-mix: indices 4-9
    symbol_to_idx.update({
        'M1': 4, 'M2': 5, 'M3': 6, 'M4': 7, 'M5': 8, 'M6': 9
    })
    
    # Three-mix: indices 10-13
    if mode in ["2mix_3mix", "2mix_3mix_4mix"]:
        symbol_to_idx.update({
            'T1': 10, 'T2': 11, 'T3': 12, 'T4': 13
        })
    
    # Four-mix: index 14
    if mode == "2mix_3mix_4mix":
        symbol_to_idx.update({
            'Q1': 14
        })
    
    return symbol_to_idx


def build_ideal_vectors(mode):
    """Build ideal frequency vectors for all symbols."""
    # [A_prob, C_prob, G_prob, T_prob]
    
    ideal_vectors = [
        # Pure bases (indices 0-3)
        [1.0, 0.0, 0.0, 0.0],  # A
        [0.0, 1.0, 0.0, 0.0],  # C
        [0.0, 0.0, 1.0, 0.0],  # G
        [0.0, 0.0, 0.0, 1.0],  # T
        
        # Two-mix (indices 4-9) - uniform 0.5/0.5
        [0.5, 0.0, 0.0, 0.5],  # M1 (A|T)
        [0.0, 0.5, 0.5, 0.0],  # M2 (C|G)
        [0.0, 0.5, 0.0, 0.5],  # M3 (C|T)
        [0.0, 0.0, 0.5, 0.5],  # M4 (G|T)
        [0.5, 0.5, 0.0, 0.0],  # M5 (A|C)
        [0.5, 0.0, 0.5, 0.0],  # M6 (A|G)
    ]
    
    # Three-mix (indices 10-13) - uniform 1/3 each
    if mode in ["2mix_3mix", "2mix_3mix_4mix"]:
        third = 1.0 / 3.0
        ideal_vectors.extend([
            [third, third, third, 0.0],    # T1 (A|C|G)
            [third, third, 0.0, third],    # T2 (A|C|T)
            [third, 0.0, third, third],    # T3 (A|G|T)
            [0.0, third, third, third],    # T4 (C|G|T)
        ])
    
    # Four-mix (index 14) - uniform 0.25 each
    if mode == "2mix_3mix_4mix":
        ideal_vectors.append(
            [0.25, 0.25, 0.25, 0.25]       # Q1 (A|C|G|T)
        )
    
    return ideal_vectors


# Build mappings for current mode
COMPOSITE_MAP = build_composite_map(CONFIG["alphabet_mode"])
SYMBOL_TO_IDX = build_symbol_to_idx(CONFIG["alphabet_mode"])
IDEAL_VECTORS = build_ideal_vectors(CONFIG["alphabet_mode"])
ALL_SYMBOLS = list(COMPOSITE_MAP.keys())

print(f"\n🧬 Composite Alphabet ({CONFIG['alphabet_mode']}):")
print(f"   Total Symbols: {len(ALL_SYMBOLS)}")
print(f"\n   {'Symbol':<8} {'Index':<6} {'Composition':<15} {'Ideal Vector [A,C,G,T]'}")
print(f"   {'-'*60}")
for sym in ALL_SYMBOLS:
    idx = SYMBOL_TO_IDX[sym]
    bases = ' | '.join(COMPOSITE_MAP[sym])
    vec = IDEAL_VECTORS[idx]
    vec_str = f"[{vec[0]:.3f}, {vec[1]:.3f}, {vec[2]:.3f}, {vec[3]:.3f}]"
    print(f"   {sym:<8} {idx:<6} {bases:<15} {vec_str}")


🧬 Composite Alphabet (2mix_only):
   Total Symbols: 10

   Symbol   Index  Composition     Ideal Vector [A,C,G,T]
   ------------------------------------------------------------
   A        0      A               [1.000, 0.000, 0.000, 0.000]
   C        1      C               [0.000, 1.000, 0.000, 0.000]
   G        2      G               [0.000, 0.000, 1.000, 0.000]
   T        3      T               [0.000, 0.000, 0.000, 1.000]
   M1       4      A | T           [0.500, 0.000, 0.000, 0.500]
   M2       5      C | G           [0.000, 0.500, 0.500, 0.000]
   M3       6      C | T           [0.000, 0.500, 0.000, 0.500]
   M4       7      G | T           [0.000, 0.000, 0.500, 0.500]
   M5       8      A | C           [0.500, 0.500, 0.000, 0.000]
   M6       9      A | G           [0.500, 0.000, 0.500, 0.000]


In [5]:
# =============================================================================
# CELL 5: ERROR RATES CLASS - EXTENDED VERSION
# =============================================================================

class ErrorRates:
    """Error rate configuration for multiple DNA sequencing technologies."""
    
    def __init__(self):
        self.general_errors = {'d': 0.0, 'ld': 0.0, 'i': 0.0, 's': 0.0}
        self.per_base_errors = {
            'A': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
            'C': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
            'G': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
            'T': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0}
        }
    
    def set_EZ17_values(self):
        """Erlich & Zielinski 2017 (Illumina MiSeq) error profile."""
        print("   >> Loading Erlich (EZ17) Error Profile...")
        # General errors
        self.general_errors = {
            's': 1.32e-03,
            'i': 5.81e-04,
            'd': 9.58e-04,
            'ld': 2.33e-04
        }
        # Per-base errors
        self.per_base_errors['A'] = {'s': 0.00135, 'i': 0.00057, 'd': 0.00099, 'ld': 0.00024}
        self.per_base_errors['C'] = {'s': 0.00135, 'i': 0.00059, 'd': 0.00098, 'ld': 0.00023}
        self.per_base_errors['G'] = {'s': 0.00126, 'i': 0.00059, 'd': 0.00094, 'ld': 0.00023}
        self.per_base_errors['T'] = {'s': 0.00132, 'i': 0.00058, 'd': 0.00096, 'ld': 0.00023}
    
    def set_G15_values(self):
        """Grass et al. 2015 (Illumina MiSeq + CustomArray) error profile."""
        print("   >> Loading Grass (G15) Error Profile...")
        # General errors
        self.general_errors = {
            's': 5.84e-03,
            'i': 8.57e-04,
            'd': 5.37e-03,
            'ld': 3.48e-04
        }
        # Per-base errors
        self.per_base_errors['A'] = {'s': 0.00605, 'i': 0.0009, 'd': 0.00543, 'ld': 0.00036}
        self.per_base_errors['C'] = {'s': 0.00563, 'i': 0.00083, 'd': 0.00513, 'ld': 0.00034}
        self.per_base_errors['G'] = {'s': 0.00577, 'i': 0.00085, 'd': 0.00539, 'ld': 0.00034}
        self.per_base_errors['T'] = {'s': 0.00591, 'i': 0.00084, 'd': 0.00559, 'ld': 0.00036}
    
    def set_O17_values(self):
        """Organick et al. 2017 (Illumina NextSeq + Twist) error profile."""
        print("   >> Loading Organick (O17) Error Profile...")
        # General errors
        self.general_errors = {
            's': 2.52e-03,
            'i': 4.14e-04,
            'd': 6.94e-04,
            'ld': 2.11e-04
        }
        # Per-base errors (converted from percentage format in original)
        self.per_base_errors['A'] = {'s': 0.00717, 'i': 0.0003, 'd': 0.00201, 'ld': 0.00054}
        self.per_base_errors['C'] = {'s': 0.00034, 'i': 0.00007, 'd': 0.00006, 'ld': 0.00001}
        self.per_base_errors['G'] = {'s': 0.00196, 'i': 0.00125, 'd': 0.00058, 'ld': 0.00023}
        self.per_base_errors['T'] = {'s': 0.00055, 'i': 0.00006, 'd': 0.00014, 'ld': 0.00006}
    
    def set_values_by_model(self, model_name):
        """Set error values based on model name."""
        if model_name == "erlich":
            self.set_EZ17_values()
        elif model_name == "grass":
            self.set_G15_values()
        elif model_name == "organick":
            self.set_O17_values()
        else:
            raise ValueError(f"Unknown error model: {model_name}")
    
    def print_current_values(self):
        print("\n   --- Error Configuration ---")
        print(f"   General: {self.general_errors}")
        for base in ['A', 'C', 'G', 'T']:
            rates = self.per_base_errors[base]
            print(f"   {base}: sub={rates['s']:.5f}, ins={rates['i']:.5f}, del={rates['d']:.5f}")
        print("   " + "-"*30)

In [6]:

# =============================================================================
# CELL 6: SEQUENCE GENERATION FUNCTIONS
# =============================================================================

def generate_composite_sequence(length, composite_map):
    """Generates a random sequence of composite symbols."""
    symbols = list(composite_map.keys())
    return [random.choice(symbols) for _ in range(length)]


def realize_sequence(composite_seq, composite_map):
    """
    Converts composite symbols to a single DNA realization.
    Each composite symbol is realized by uniformly picking one of its nucleotides.
    """
    realized = []
    for sym in composite_seq:
        nucleotide = random.choice(composite_map[sym])
        realized.append(nucleotide)
    return "".join(realized)


def apply_ids_noise(sequence, error_profile):
    """
    Apply Insertion, Deletion, Substitution noise to a DNA sequence.
    
    Args:
        sequence: Clean DNA string
        error_profile: ErrorRates object with per-base error rates
    
    Returns:
        Noisy DNA string
    """
    bases = ['A', 'C', 'G', 'T']
    noisy_seq = []
    
    for base in sequence:
        if base not in bases:
            continue
        
        rates = error_profile.per_base_errors[base]
        p_sub = rates['s']
        p_ins = rates['i']
        p_del = rates['d']
        
        # 1. DELETION Check
        if random.random() < p_del:
            continue  # Skip this base (deletion)
            
        # 2. INSERTION Check (pre-insertion)
        if random.random() < p_ins:
            noisy_seq.append(random.choice(bases))
            
        # 3. SUBSTITUTION vs MATCH Check
        if random.random() < p_sub:
            options = [b for b in bases if b != base]
            noisy_seq.append(random.choice(options))
        else:
            noisy_seq.append(base)
            
    return "".join(noisy_seq)


In [7]:

# =============================================================================
# CELL 7: MAIN DATASET GENERATION FUNCTION
# =============================================================================

def generate_dataset(config, composite_map, symbol_to_idx, ideal_vectors):
    """
    Generate dataset with extended composite alphabet.
    
    Creates a pickle file containing:
    - metadata: Configuration and generation parameters
    - data: List of samples, each with 'id', 'label' (composite sequence), 'cluster' (noisy reads)
    """
    # Setup error profile
    errors = ErrorRates()
    errors.set_EZ17_values()
    errors.print_current_values()
    
    num_samples = config['num_samples']
    seq_length = config['seq_length']
    coverage = config['max_coverage']
    filename = config['dataset_path']
    
    # Initialize dataset structure
    dataset = {
        'metadata': {
            'type': f'Composite DNA ({config["alphabet_mode"]})',
            'error_profile': 'Erlich (EZ17)',
            'num_samples': num_samples,
            'seq_length': seq_length,
            'coverage_depth': coverage,
            'vocab_size': config['vocab_size'],
            'alphabet_mode': config['alphabet_mode'],
            'symbols': list(symbol_to_idx.keys()),
            'symbol_to_idx': symbol_to_idx,
            'ideal_vectors': ideal_vectors,
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'seed': config['seed']
        },
        'data': []
    }
    
    print(f"\n{'='*60}")
    print(f"🔄 GENERATING DATASET")
    print(f"{'='*60}")
    print(f"   Samples: {num_samples:,}")
    print(f"   Sequence Length: {seq_length}")
    print(f"   Coverage Depth: {coverage}")
    print(f"   Alphabet: {config['alphabet_mode']} ({config['vocab_size']} classes)")
    print(f"{'='*60}")
    
    start_time = time.time()
    
    for i in range(num_samples):
        # A. Generate Ground Truth (Label) - composite sequence
        clean_composite_seq = generate_composite_sequence(seq_length, composite_map)
        
        # B. Generate Cluster (Input) - multiple noisy reads
        cluster_reads = []
        for _ in range(coverage):
            # 1. Realize: Composite -> DNA
            realized_dna = realize_sequence(clean_composite_seq, composite_map)
            # 2. Corrupt: DNA -> Noisy DNA
            noisy_read = apply_ids_noise(realized_dna, errors)
            cluster_reads.append(noisy_read)
            
        # C. Store sample
        sample = {
            'id': i,
            'label': clean_composite_seq,
            'cluster': cluster_reads
        }
        dataset['data'].append(sample)
        
        # Progress logging
        if (i + 1) % 10000 == 0:
            elapsed = time.time() - start_time
            samples_per_sec = (i + 1) / elapsed
            eta = (num_samples - i - 1) / samples_per_sec
            print(f"   Processed {i+1:,}/{num_samples:,} | "
                  f"Speed: {samples_per_sec:.1f} samples/s | "
                  f"ETA: {eta:.1f}s")

    # Save dataset
    with open(filename, 'wb') as f:
        pickle.dump(dataset, f)
    
    total_time = time.time() - start_time
    
    print(f"\n{'='*60}")
    print(f"✅ DATASET GENERATION COMPLETE")
    print(f"{'='*60}")
    print(f"   Output File: {filename}")
    print(f"   Total Time: {total_time:.1f}s ({total_time/60:.1f} min)")
    print(f"   File Size: {os.path.getsize(filename) / (1024*1024):.1f} MB")
    
    # Print example
    print(f"\n📝 Sample 0:")
    print(f"   Label (first 10): {dataset['data'][0]['label'][:10]}")
    print(f"   Read 1 (first 20): {dataset['data'][0]['cluster'][0][:20]}...")
    
    return dataset

In [8]:

# =============================================================================
# CELL 8: ANALYZE DATASET STATISTICS
# =============================================================================

def analyze_dataset(dataset, symbol_to_idx):
    """Analyze and print dataset statistics."""
    
    print(f"\n{'='*60}")
    print(f"📊 DATASET STATISTICS")
    print(f"{'='*60}")
    
    # Symbol distribution in labels
    all_symbols = []
    for sample in dataset['data']:
        all_symbols.extend(sample['label'])
    
    counter = Counter(all_symbols)
    total_symbols = len(all_symbols)
    
    print(f"\n🧬 Symbol Distribution in Labels:")
    print(f"   {'Symbol':<8} {'Count':>10} {'Percentage':>12} {'Expected':>10}")
    print(f"   {'-'*42}")
    
    num_classes = len(symbol_to_idx)
    expected_pct = 100.0 / num_classes
    
    for sym in sorted(counter.keys(), key=lambda x: symbol_to_idx[x]):
        count = counter[sym]
        pct = 100 * count / total_symbols
        print(f"   {sym:<8} {count:>10,} {pct:>11.2f}% {expected_pct:>9.2f}%")
    
    print(f"   {'-'*42}")
    print(f"   {'Total':<8} {total_symbols:>10,}")
    
    # Read length statistics
    all_read_lengths = []
    for sample in dataset['data']:
        for read in sample['cluster']:
            all_read_lengths.append(len(read))
    
    print(f"\n📏 Read Length Statistics:")
    print(f"   Original Length: {dataset['metadata']['seq_length']}")
    print(f"   Mean Read Length: {np.mean(all_read_lengths):.2f}")
    print(f"   Std Read Length: {np.std(all_read_lengths):.2f}")
    print(f"   Min Read Length: {np.min(all_read_lengths)}")
    print(f"   Max Read Length: {np.max(all_read_lengths)}")
    
    # Symbol type distribution
    pure_count = sum(counter.get(s, 0) for s in ['A', 'C', 'G', 'T'])
    two_mix_count = sum(counter.get(f'M{i}', 0) for i in range(1, 7))
    three_mix_count = sum(counter.get(f'T{i}', 0) for i in range(1, 5))
    four_mix_count = counter.get('Q1', 0)
    
    print(f"\n📈 Symbol Type Distribution:")
    print(f"   Pure Bases (A,C,G,T): {pure_count:,} ({100*pure_count/total_symbols:.1f}%)")
    print(f"   Two-Mix (M1-M6):      {two_mix_count:,} ({100*two_mix_count/total_symbols:.1f}%)")
    if three_mix_count > 0:
        print(f"   Three-Mix (T1-T4):    {three_mix_count:,} ({100*three_mix_count/total_symbols:.1f}%)")
    if four_mix_count > 0:
        print(f"   Four-Mix (Q1):        {four_mix_count:,} ({100*four_mix_count/total_symbols:.1f}%)")


In [9]:


# =============================================================================
# CELL 9: MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    
    print("\n" + "="*60)
    print(f"🧬 COMPOSITE DNA DATASET GENERATOR - {CONFIG['error_model'].upper()}")
    print(f"   Error Model: {CONFIG['error_specs']['name']}")
    print(f"   Sequence Length: {CONFIG['seq_length']}")
    print(f"   Alphabet: {CONFIG['alphabet_mode']}")
    print("="*60)
    
    # Check if dataset already exists
    if os.path.exists(CONFIG['dataset_path']):
        print(f"\n⚠️  Dataset already exists: {CONFIG['dataset_path']}")
        response = input("   Overwrite? (y/n): ").strip().lower()
        if response != 'y':
            print("   Aborted.")
            exit()
    
    # Build mappings
    COMPOSITE_MAP = build_composite_map(CONFIG["alphabet_mode"])
    SYMBOL_TO_IDX = build_symbol_to_idx(CONFIG["alphabet_mode"])
    IDEAL_VECTORS = build_ideal_vectors(CONFIG["alphabet_mode"])
    ALL_SYMBOLS = list(COMPOSITE_MAP.keys())
    
    # Generate dataset
    dataset = generate_dataset(CONFIG, COMPOSITE_MAP, SYMBOL_TO_IDX, IDEAL_VECTORS)
    
    # Analyze dataset
    analyze_dataset(dataset, SYMBOL_TO_IDX)
    
    print(f"\n{'='*60}")
    print(f"🎉 Dataset generation completed successfully!")
    print(f"   Error Model: {CONFIG['error_model'].upper()}")
    print(f"   File: {CONFIG['dataset_path']}")
    print(f"{'='*60}")


🧬 COMPOSITE DNA DATASET GENERATOR - ERLICH
   Error Model: EZ17
   Sequence Length: 136
   Alphabet: 2mix_only
   >> Loading Erlich (EZ17) Error Profile...

   --- Error Configuration ---
   General: {'s': 0.00132, 'i': 0.000581, 'd': 0.000958, 'ld': 0.000233}
   A: sub=0.00135, ins=0.00057, del=0.00099
   C: sub=0.00135, ins=0.00059, del=0.00098
   G: sub=0.00126, ins=0.00059, del=0.00094
   T: sub=0.00132, ins=0.00058, del=0.00096
   ------------------------------

🔄 GENERATING DATASET
   Samples: 100,000
   Sequence Length: 136
   Coverage Depth: 25
   Alphabet: 2mix_only (10 classes)
   Processed 10,000/100,000 | Speed: 297.4 samples/s | ETA: 302.6s
   Processed 20,000/100,000 | Speed: 299.5 samples/s | ETA: 267.1s
   Processed 30,000/100,000 | Speed: 299.9 samples/s | ETA: 233.4s
   Processed 40,000/100,000 | Speed: 300.4 samples/s | ETA: 199.7s
   Processed 50,000/100,000 | Speed: 299.0 samples/s | ETA: 167.2s
   Processed 60,000/100,000 | Speed: 299.5 samples/s | ETA: 133.6s
  